# Week 2: Embedding Pipeline & Classical Baseline (N=1000)
**Project**: Quantum Kernels on CLIP / BERT Embeddings

**What this notebook does:**
1. Loads SST-2 sentiment dataset from HuggingFace
2. Extracts BERT embeddings (768-dimensional [CLS] vectors)
3. Reduces to 8 dimensions via PCA for quantum encoding
4. Normalizes to [0, pi] with defensive clipping
5. Trains 3 classical SVM baselines on a real 1000-train / 200-test split
6. Saves everything for Week 3 (quantum kernel experiments)

**This version produces a genuine N=1000 training subset.**

**Runtime**: ~12 minutes on a free Colab T4 (BERT extraction is the slow step)

In [ ]:
!pip install datasets transformers torch scikit-learn numpy --quiet

import numpy as np
import time, os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("All imports OK")
print(f"PyTorch: {torch.__version__} | GPU available: {torch.cuda.is_available()}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

---
## Step 2: Load SST-2 dataset

SST-2 (Stanford Sentiment Treebank) is the standard binary sentiment benchmark.
Label 0 = negative, Label 1 = positive.

We extract 1100 train + 250 validation so we can build a clean 1000 / 200 split later.

**Note**: the SST-2 validation set has only 872 samples total, so keep N_VAL_EXTRACT below that.

In [ ]:
# Load SST-2 from HuggingFace
print("Loading SST-2 dataset...")
dataset = load_dataset("nyu-mll/glue", "sst2")   # namespaced path ('glue' alone is broken now)

# Extract enough to build a real 1000-train / 200-test set
N_TRAIN_EXTRACT = 1100   # buffer above 1000
N_VAL_EXTRACT   = 250    # buffer above 200 (max ~870 available)

train_data = dataset['train'].shuffle(seed=42).select(range(N_TRAIN_EXTRACT))
val_data   = dataset['validation'].shuffle(seed=42).select(range(N_VAL_EXTRACT))

print(f"Train split: {len(train_data)} samples")
print(f"  Negative: {sum(1 for x in train_data if x['label'] == 0)}")
print(f"  Positive: {sum(1 for x in train_data if x['label'] == 1)}")
print(f"\nValidation split: {len(val_data)} samples")
print(f"\nExample sentences:")
for i in range(3):
    label_name = 'positive' if train_data[i]['label'] == 1 else 'negative'
    print(f"  [{label_name}] {train_data[i]['sentence'][:80]}...")

---
## Step 3: Extract BERT embeddings

We use `bert-base-uncased` and take the **[CLS] token embedding** for each sentence —
a 768-dimensional vector that aggregates sentence-level meaning.

We use a *frozen, pretrained* model intentionally: the research question is whether
quantum kernels add value on top of general-purpose embeddings, so the encoder stays fixed.

In [ ]:
MODEL_NAME = "bert-base-uncased"
print(f"Loading {MODEL_NAME}...")

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()

print(f"Model loaded. Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")

def extract_bert_embeddings(sentences, batch_size=32, show_progress=True):
    """Extract [CLS] token embeddings from BERT for a list of sentences."""
    all_embeddings = []
    n_batches = (len(sentences) + batch_size - 1) // batch_size
    for i in range(0, len(sentences), batch_size):
        batch  = sentences[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            outputs = bert_model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]   # [CLS] token
        all_embeddings.append(cls_embeddings.cpu().numpy())
        if show_progress and (i // batch_size + 1) % 5 == 0:
            print(f"  Batch {i // batch_size + 1}/{n_batches} done...")
    return np.vstack(all_embeddings)

print("\nExtracting train embeddings (~1100 sentences)...")
t0 = time.time()
train_sentences = [x['sentence'] for x in train_data]
train_labels    = np.array([x['label'] for x in train_data])
train_emb = extract_bert_embeddings(train_sentences)
print(f"  Done in {time.time()-t0:.1f}s | Shape: {train_emb.shape}")

print("\nExtracting validation embeddings (~250 sentences)...")
t0 = time.time()
val_sentences = [x['sentence'] for x in val_data]
val_labels    = np.array([x['label'] for x in val_data])
val_emb = extract_bert_embeddings(val_sentences)
print(f"  Done in {time.time()-t0:.1f}s | Shape: {val_emb.shape}")

print(f"\nEmbedding stats:")
print(f"  Mean L2 norm: {np.mean(np.linalg.norm(train_emb, axis=1)):.3f}")

---
## Step 4: PCA reduction

Quantum angle encoding maps one feature per qubit. We reduce 768 -> 8 dimensions.
We also fit 4, 12, 16 for the Week 4 ablation study.

**Important**: fit PCA on training data only, then transform both sets.

In [ ]:
N_QUBITS = 8
pca_dims_to_try = [4, 8, 12, 16]

print("PCA variance explained by dimension:")
print(f"  {'Dims':>6}  {'Var Explained':>15}")
print(f"  {'-'*25}")

pca_objects = {}
for n in pca_dims_to_try:
    pca = PCA(n_components=n, random_state=42)
    pca.fit(train_emb)
    var = np.sum(pca.explained_variance_ratio_) * 100
    pca_objects[n] = pca
    print(f"  {n:>6}  {var:>14.1f}%")

pca_8       = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb)
X_val_pca   = pca_8.transform(val_emb)

print(f"\nPrimary reduction: 768 -> {N_QUBITS} dims")
print(f"  Train shape: {X_train_pca.shape}")
print(f"  Val   shape: {X_val_pca.shape}")

# Cumulative variance plot
plt.figure(figsize=(8, 3))
pca_plot = PCA(n_components=50, random_state=42).fit(train_emb)
plt.plot(np.cumsum(pca_plot.explained_variance_ratio_) * 100, color='steelblue')
plt.axvline(x=N_QUBITS-1, color='coral', linestyle='--', label=f'{N_QUBITS} dims (used)')
plt.axhline(y=80, color='gray', linestyle=':', alpha=0.5, label='80% variance')
plt.xlabel('Number of PCA components'); plt.ylabel('Cumulative variance (%)')
plt.title('BERT embedding PCA: cumulative variance')
plt.legend(); plt.tight_layout(); plt.show()

---
## Step 5: Normalize to [0, pi] for quantum encoding

Quantum rotation gates take angles in [0, 2pi]; we use [0, pi] by convention.

**Critical fix**: fit the scaler on train only, then clip validation data to [0, pi].
Validation values can fall slightly outside the training range, which would silently
corrupt quantum rotation gates in Week 3.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)

# Transform val with train statistics, then clip out-of-range values
X_val_scaled = scaler.transform(X_val_pca)
X_val_scaled = np.clip(X_val_scaled, 0, np.pi)   # <-- defensive clip

print("After normalization:")
print(f"  Train range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")
print(f"  Val   range: [{X_val_scaled.min():.4f}, {X_val_scaled.max():.4f}]")
print(f"  (train is exactly [0, pi]; val clipped to safe range)")

---
## Step 6: Classical SVM baselines on the real 1000 / 200 split

We build a genuine 1000-train / 200-test subset and train 3 SVM kernels.
These are the benchmarks the quantum kernel competes with in Week 3.

In [ ]:
# Build the real 1000-train / 200-test subset
N_TRAIN_SVM = 1000
N_TEST_SVM  = 200

# Safety check — make sure we extracted enough
assert len(X_train_scaled) >= N_TRAIN_SVM, f"Only {len(X_train_scaled)} train available, need {N_TRAIN_SVM}"
assert len(X_val_scaled)   >= N_TEST_SVM,  f"Only {len(X_val_scaled)} val available, need {N_TEST_SVM}"

idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), N_TRAIN_SVM, replace=False)
idx_te = np.random.RandomState(42).choice(len(X_val_scaled),   N_TEST_SVM,  replace=False)

X_tr = X_train_scaled[idx_tr]
y_tr = train_labels[idx_tr]
X_te = X_val_scaled[idx_te]
y_te = val_labels[idx_te]

print(f"SVM training on {len(X_tr)} samples, testing on {len(X_te)} samples")
print(f"Class balance: train {np.mean(y_tr==1):.1%} pos | test {np.mean(y_te==1):.1%} pos")
print()

classifiers = {
    'Linear':      SVC(kernel='linear',  C=1.0, random_state=42),
    'RBF':         SVC(kernel='rbf',     C=1.0, gamma='scale', random_state=42),
    'Polynomial':  SVC(kernel='poly',    C=1.0, degree=3, coef0=1, random_state=42),
}

results = {}
print(f"{'Kernel':<14} {'Accuracy':>10} {'F1':>8} {'Train Time':>12}")
print(f"{'-'*48}")
for name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    train_time = time.time() - t0
    y_pred = clf.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred)
    results[name] = {'accuracy': acc, 'f1': f1, 'train_time': train_time, 'clf': clf}
    print(f"{name:<14} {acc:>10.4f} {f1:>8.4f} {train_time:>10.3f}s")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]
f1s   = [results[n]['f1'] for n in names]
axes[0].bar(names, accs, color=['steelblue', 'coral', 'mediumseagreen'])
axes[0].set_ylim(0.5, 1.0); axes[0].set_title('Accuracy by kernel'); axes[0].set_ylabel('Accuracy')
for i, v in enumerate(accs): axes[0].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11)
axes[1].bar(names, f1s, color=['steelblue', 'coral', 'mediumseagreen'])
axes[1].set_ylim(0.5, 1.0); axes[1].set_title('F1 by kernel'); axes[1].set_ylabel('F1 score')
for i, v in enumerate(f1s): axes[1].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11)
plt.suptitle('Classical baselines on 8-dim BERT embeddings (SST-2, N=1000)')
plt.tight_layout(); plt.show()

---
## Step 7: Effect of PCA dimensions

Shows how much accuracy changes at 4, 8, 12, 16 dims — guides Week 4 qubit ablation.

In [ ]:
print("RBF SVM accuracy vs PCA dimensions (train=1000, test=200)")
print()
print(f"  {'Dims':>6}  {'Var Explained':>14}  {'Accuracy':>10}  {'F1':>8}")
print(f"  {'-'*46}")
dim_results = {}
for n_dim in pca_dims_to_try:
    pca_d  = pca_objects[n_dim]
    X_tr_d = pca_d.transform(train_emb[idx_tr])
    X_te_d = pca_d.transform(val_emb[idx_te])
    sc_d   = MinMaxScaler(feature_range=(0, np.pi))
    X_tr_d = sc_d.fit_transform(X_tr_d)
    X_te_d = np.clip(sc_d.transform(X_te_d), 0, np.pi)
    clf_d  = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(X_tr_d, y_tr)
    y_pred_d = clf_d.predict(X_te_d)
    acc_d = accuracy_score(y_te, y_pred_d)
    f1_d  = f1_score(y_te, y_pred_d)
    var_d = np.sum(pca_objects[n_dim].explained_variance_ratio_) * 100
    dim_results[n_dim] = acc_d
    print(f"  {n_dim:>6}  {var_d:>13.1f}%  {acc_d:>10.4f}  {f1_d:>8.4f}")

---
## Step 8: Save everything for Week 3

Saves the real 1000 / 200 split plus PCA and scaler objects.

In [ ]:
import pickle
os.makedirs('qml_project', exist_ok=True)

np.savez('qml_project/embeddings_week2.npz',
         X_train=X_train_scaled, X_val=X_val_scaled,
         y_train=train_labels,   y_val=val_labels,
         X_tr_subset=X_tr, X_te_subset=X_te,
         y_tr_subset=y_tr, y_te_subset=y_te)

with open('qml_project/pca_scaler.pkl', 'wb') as f:
    pickle.dump({'pca': pca_8, 'scaler': scaler, 'n_qubits': N_QUBITS}, f)

baseline_summary = {k: {'accuracy': v['accuracy'], 'f1': v['f1']} for k, v in results.items()}
with open('qml_project/baseline_results.pkl', 'wb') as f:
    pickle.dump(baseline_summary, f)

# Verify reload AND shapes
loaded = np.load('qml_project/embeddings_week2.npz')
print("Files saved. Verifying shapes:")
print(f"  X_tr_subset = {loaded['X_tr_subset'].shape}  (must be (1000, 8))")
print(f"  X_te_subset = {loaded['X_te_subset'].shape}  (must be (200, 8))")
assert loaded['X_tr_subset'].shape == (1000, 8), "TRAIN SUBSET WRONG SIZE"
assert loaded['X_te_subset'].shape == (200, 8),  "TEST SUBSET WRONG SIZE"
print("\n  VERIFIED: genuine N=1000 / 200 split saved correctly.")

print("\nFiles in qml_project/:")
for fname in os.listdir('qml_project'):
    sz = os.path.getsize(f'qml_project/{fname}') / 1024
    print(f"  qml_project/{fname}  ({sz:.0f} KB)")

# Download immediately — Colab sessions die without warning
from google.colab import files
print("\nDownloading files now (don't wait for session to expire)...")
files.download('qml_project/embeddings_week2.npz')
files.download('qml_project/baseline_results.pkl')
files.download('qml_project/pca_scaler.pkl')

---
## Week 2 complete

### What you built:
- Real 1000-train / 200-test split from SST-2 BERT embeddings
- Classical SVM baselines (linear, RBF, polynomial) at genuine N=1000
- All files saved AND downloaded

### Sanity check before Week 3:
The shape assertions in Step 8 must pass — X_tr_subset = (1000, 8). If they fail,
you didn't extract enough data; increase N_TRAIN_EXTRACT in Step 2.

### What changes from the old 500-sample run:
Your baseline accuracies will shift slightly (the old 0.67 Linear was on 500 samples).
This is expected — bigger, cleaner sample.

### Week 3:
The genuine N=1000 quantum kernel matrix will take ~83 min to compute
(1,000,000 evaluations), versus ~23 min for the old 500-sample version.
That time difference confirms you're really at N=1000.